# CommerceOps-Env — GRPO Training

**Project**: CommerceOps-Env (Meta PyTorch OpenEnv Hackathon Grand Finale)  
**Claim**: An LLM can be trained with GRPO to make better human-like fulfillment decisions under inventory scarcity, SLA pressure, and competing customer priorities.

## Notebook structure
1. Install dependencies
2. Clone / mount the environment
3. Measure **baseline** (frozen Qwen2.5-3B-Instruct, no training)
4. Run a **tiny GRPO experiment** (few hundred steps, T1 + T2)
5. Measure **post-training** performance
6. Plot reward curves and compare before/after

> **Runtime**: T4 GPU in Colab. Full GRPO run takes ~30–60 min.  
> Set `FAST_DEV_RUN = True` (cell 3) to do a 5-step smoke test first.

## 1 · Install dependencies

In [ ]:
# Unsloth speeds up GRPO fine-tuning significantly on consumer GPUs.
# TRL >= 0.12 has native GRPOTrainer.
!pip install -q unsloth trl>=0.12 peft accelerate bitsandbytes datasets
!pip install -q fastapi uvicorn pydantic httpx

# Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2 · Mount environment

In [ ]:
import sys, os

# ── Option A: clone from GitHub (replace with your repo URL) ──────────
REPO_URL = 'https://github.com/YOUR_USERNAME/ecommerce-ops-env-starter.git'
REPO_DIR = '/content/ecommerce-ops-env-starter'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

# ── Option B: mount from Google Drive ────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_DIR = '/content/drive/MyDrive/ecommerce-ops-env-starter'

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Quick sanity check
from environment import CommerceOpsEnv
from tasks import get_task_bundle
env = CommerceOpsEnv()
obs = env.reset('task_1', seed=0)
print('Env OK — task_id:', obs.task_id, '| orders:', len(obs.orders))

## 3 · Config

In [ ]:
# ─── EDIT THESE ──────────────────────────────────────────────────────
FAST_DEV_RUN   = False    # True = 5-step smoke test, no real training
HF_TOKEN       = ''       # set to push model to Hub (optional)
PUSH_TO_HUB    = False    # only if HF_TOKEN is set
HUB_REPO_ID    = 'YOUR_HF_USERNAME/commerce-ops-grpo'
# ─────────────────────────────────────────────────────────────────────

MODEL_NAME     = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN    = 2048
LORA_R         = 16
LORA_ALPHA     = 32

# Training
NUM_TRAIN_STEPS = 5 if FAST_DEV_RUN else 200   # 200 steps is minimal but measurable
BATCH_SIZE      = 2
GRPO_N_SAMPLES  = 4    # rollouts per prompt (GRPO needs >1)
LR              = 5e-6

# Eval
EVAL_SEEDS      = list(range(8))    # 8 deterministic seeds for stable avg
TASKS_TO_EVAL   = ['task_1', 'task_2']

print(f'Model: {MODEL_NAME}')
print(f'Steps: {NUM_TRAIN_STEPS} | batch: {BATCH_SIZE} | rollouts/prompt: {GRPO_N_SAMPLES}')
print(f'FAST_DEV_RUN: {FAST_DEV_RUN}')

## 4 · Prompt + action parsing utilities

In [ ]:
import json, re
from typing import Any, Dict, List, Optional


SYSTEM_PROMPT = """\
You are a fulfillment operations manager for an e-commerce platform.
You receive structured observations about pending orders, warehouse stock, 
shipping constraints, customer tiers, and SLA deadlines.

Your job is to decide the BEST action for the current situation.
Respond with a single JSON object matching the action schema.
Do NOT include any explanation outside the JSON.

Action schema:
  {"action_type": "<type>", ...fields}

Allowed action_type values (vary by task):
  assign_warehouse   — {"action_type": "assign_warehouse", "order_id": "...", "warehouse_id": "..."}
  split_shipment     — {"action_type": "split_shipment", "order_id": "...", "allocations": [{"warehouse_id": "...", "quantity": N}, ...]}
  delay_order        — {"action_type": "delay_order", "order_id": "...", "reason": "..."}
  prioritize_order   — {"action_type": "prioritize_order", "order_id": "..."}
  noop               — {"action_type": "noop"}
"""


def obs_to_prompt(obs) -> str:
    """Convert an EnvObservation to a compact JSON-in-text prompt."""
    snap = {
        'task': obs.task_id,
        'step': obs.step,
        'steps_remaining': obs.steps_remaining,
        'description': obs.task_description,
        'policy_flags': obs.policy_flags,
        'allowed_actions': [a.value if hasattr(a, 'value') else a for a in obs.allowed_actions],
        'orders': [o.model_dump() for o in obs.orders],
        'warehouses': [w.model_dump() for w in obs.warehouses],
        'stock': [s.model_dump() for s in obs.stock],
    }
    return json.dumps(snap, indent=2)


def build_messages(obs) -> List[Dict[str, str]]:
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_to_prompt(obs)},
    ]


# JSON extraction — try strict then regex fallback
_JSON_RE = re.compile(r'\{[^{}]*\}', re.DOTALL)

def extract_action(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()
    # Strip markdown code fences
    text = re.sub(r'^```(?:json)?\n?', '', text)
    text = re.sub(r'\n?```$', '', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = _JSON_RE.search(text)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


print('Utilities loaded.')

## 5 · Rollout runner

In [ ]:
import torch
from transformers import AutoTokenizer


def run_episode(
    model,
    tokenizer,
    task_id: str,
    seed: int = 0,
    max_new_tokens: int = 256,
    greedy: bool = True,
) -> Dict[str, Any]:
    """Run one full episode and return score + trajectory."""
    env = CommerceOpsEnv()
    obs = env.reset(task_id=task_id, seed=seed)
    trajectory = []
    total_reward = 0.0

    while not obs.done:
        messages = build_messages(obs)
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=not greedy,
                temperature=0.7 if not greedy else None,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = out[0][input_ids.shape[-1]:]
        raw_text  = tokenizer.decode(generated, skip_special_tokens=True)
        action    = extract_action(raw_text) or {'action_type': 'noop'}

        obs = env.step(action)
        total_reward += obs.reward
        trajectory.append({
            'step': obs.step,
            'raw_output': raw_text,
            'action': action,
            'reward': obs.reward,
            'error': obs.last_action_error,
        })

    result = env.final_score()
    return {
        'task_id': task_id,
        'seed': seed,
        'score': result['score'],
        'breakdown': result['breakdown'],
        'total_reward': round(total_reward, 4),
        'steps': len(trajectory),
        'invalid_actions': result['invalid_actions'],
        'trajectory': trajectory,
    }


def evaluate_model(
    model,
    tokenizer,
    tasks: List[str] = TASKS_TO_EVAL,
    seeds: List[int] = EVAL_SEEDS,
    label: str = '',
) -> Dict[str, Any]:
    """Run across tasks × seeds and return aggregate stats."""
    results = []
    for task_id in tasks:
        for seed in seeds:
            r = run_episode(model, tokenizer, task_id=task_id, seed=seed)
            results.append(r)
            print(f'  [{label}] {task_id} seed={seed} score={r["score"]:.3f} '
                  f'reward={r["total_reward"]:.3f} invalid={r["invalid_actions"]}')

    by_task = {}
    for task_id in tasks:
        task_results = [r for r in results if r['task_id'] == task_id]
        by_task[task_id] = {
            'mean_score':   round(sum(r['score']         for r in task_results) / len(task_results), 4),
            'mean_reward':  round(sum(r['total_reward']  for r in task_results) / len(task_results), 4),
            'invalid_rate': round(sum(r['invalid_actions'] for r in task_results) /
                                  max(sum(r['steps'] for r in task_results), 1), 4),
        }
    return {'label': label, 'by_task': by_task, 'all': results}


print('Rollout utilities loaded.')

## 6 · Load model (Unsloth 4-bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto
    load_in_4bit=True,
    token=HF_TOKEN or None,
)

print(f'Loaded {MODEL_NAME}')
print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 7 · Baseline (frozen instruct model)

In [ ]:
FastLanguageModel.for_inference(model)

print('=== BASELINE (frozen Qwen2.5-3B-Instruct) ===')
baseline_results = evaluate_model(model, tokenizer, label='baseline')

print('\nBaseline summary:')
for task_id, stats in baseline_results['by_task'].items():
    print(f'  {task_id}: mean_score={stats["mean_score"]:.3f}  '
          f'mean_reward={stats["mean_reward"]:.3f}  '
          f'invalid_rate={stats["invalid_rate"]:.3f}')

## 8 · Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA adapters attached.')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M')

## 9 · Build GRPO dataset from environment rollouts

In [ ]:
from datasets import Dataset


def make_grpo_prompt(task_id: str, seed: int) -> str:
    """Return the initial observation as a chat-formatted string.
    GRPOTrainer calls the reward function with (prompt, completion) pairs.
    """
    env = CommerceOpsEnv()
    obs = env.reset(task_id=task_id, seed=seed)
    messages = build_messages(obs)
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# Mix T1 (easy, for non-zero reward early) and T2 (headline task).
# 3 seeds × 2 tasks = 6 prompts; GRPOTrainer generates N_SAMPLES per prompt.
TRAIN_SEEDS = [0, 1, 2] if not FAST_DEV_RUN else [0]
TRAIN_TASKS = ['task_1', 'task_2'] if not FAST_DEV_RUN else ['task_1']

prompts = [
    {'prompt': make_grpo_prompt(tid, seed), 'task_id': tid, 'seed': seed}
    for tid in TRAIN_TASKS
    for seed in TRAIN_SEEDS
]

train_dataset = Dataset.from_list(prompts)
print(f'Training dataset: {len(train_dataset)} prompts')
print(train_dataset[0]['prompt'][:300], '...')

## 10 · Reward function for GRPOTrainer

In [ ]:
from typing import List as TList


def grpo_reward_fn(prompts: TList[str], completions: TList[str], **kwargs) -> TList[float]:
    """Reward function called by GRPOTrainer.

    Each call receives a batch of (prompt, completion) pairs for the SAME
    episode initial state. We re-instantiate the env from the metadata in
    ``kwargs`` (task_id, seed) and run the full episode using the completion
    as the first action. Single-step reward is returned for GRPOTrainer;
    we use the *episode-level score* so the signal is outcome-based, not
    token-level.
    """
    task_ids = kwargs.get('task_id', ['task_1'] * len(completions))
    seeds    = kwargs.get('seed',    [0]        * len(completions))

    rewards = []
    for completion, task_id, seed in zip(completions, task_ids, seeds):
        action = extract_action(completion) or {'action_type': 'noop'}
        try:
            env = CommerceOpsEnv()
            env.reset(task_id=task_id, seed=int(seed))
            obs = env.step(action)
            # Use step reward + partial episode score for a richer signal.
            episode_score = env.final_score()['score']
            reward = float(obs.reward) + episode_score * 0.5
        except Exception:
            reward = -0.5   # malformed output
        rewards.append(reward)
    return rewards


# Quick sanity check
test_rewards = grpo_reward_fn(
    prompts=['dummy'],
    completions=['{"action_type": "assign_warehouse", "order_id": "O1", "warehouse_id": "W1"}'],
    task_id=['task_1'],
    seed=[0],
)
print('Reward fn sanity check (T1 best action):', test_rewards)

## 11 · GRPO training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='./grpo_output',
    max_steps=NUM_TRAIN_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=GRPO_N_SAMPLES,
    learning_rate=LR,
    max_prompt_length=1024,
    max_completion_length=256,
    logging_steps=10,
    save_steps=50,
    warmup_steps=10,
    # Keep KL penalty small — reward is mostly outcome-based
    kl_coef=0.01,
    report_to='none',   # change to 'wandb' if you have it set up
    seed=42,
)

FastLanguageModel.for_training(model)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=grpo_reward_fn,
    processing_class=tokenizer,
)

print(f'Starting GRPO training for {NUM_TRAIN_STEPS} steps...')
train_result = trainer.train()
print('Training complete!')
print('Train loss:', round(train_result.training_loss, 4))

## 12 · Post-training evaluation

In [ ]:
FastLanguageModel.for_inference(model)

print('=== POST-TRAINING ===')
trained_results = evaluate_model(model, tokenizer, label='trained')

print('\nPost-training summary:')
for task_id, stats in trained_results['by_task'].items():
    print(f'  {task_id}: mean_score={stats["mean_score"]:.3f}  '
          f'mean_reward={stats["mean_reward"]:.3f}  '
          f'invalid_rate={stats["invalid_rate"]:.3f}')

## 13 · Before / after comparison

In [ ]:
print('\n' + '='*60)
print('BEFORE vs AFTER  (mean across seeds)')
print('='*60)
print(f'{"Task":<12} {"Metric":<18} {"Baseline":>10} {"Trained":>10} {"Delta":>10}')
print('-'*60)

for task_id in TASKS_TO_EVAL:
    b = baseline_results['by_task'][task_id]
    t = trained_results['by_task'][task_id]
    for metric in ('mean_score', 'mean_reward', 'invalid_rate'):
        delta = t[metric] - b[metric]
        sign = '+' if delta >= 0 else ''
        print(f'{task_id:<12} {metric:<18} {b[metric]:>10.4f} {t[metric]:>10.4f} {sign+str(round(delta,4)):>10}')
print('='*60)

## 14 · Reward curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Parse trainer log
log_history = trainer.state.log_history
df = pd.DataFrame([e for e in log_history if 'loss' in e or 'reward' in e])
df = df.dropna(subset=['step'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Reward curve
if 'reward' in df.columns:
    axes[0].plot(df['step'], df['reward'], marker='o', ms=3, label='mean reward')
    axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='random floor')
    axes[0].set_title('GRPO — Mean Reward per Step')
    axes[0].set_xlabel('Training step')
    axes[0].set_ylabel('Reward')
    axes[0].legend()

# Loss curve
if 'loss' in df.columns:
    axes[1].plot(df['step'], df['loss'], color='orange', marker='o', ms=3, label='loss')
    axes[1].set_title('GRPO — Training Loss')
    axes[1].set_xlabel('Training step')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

plt.tight_layout()
plt.savefig('grpo_curves.png', dpi=150)
plt.show()
print('Saved: grpo_curves.png')

## 15 · Sample rollout inspection (anti-hacking check)

In [ ]:
# Inspect one T2 rollout from the trained model to make sure
# the reward increase comes from better decisions, not format gaming.

sample = run_episode(model, tokenizer, task_id='task_2', seed=0)
print(f'T2 seed=0  score={sample["score"]:.3f}  '
      f'total_reward={sample["total_reward"]:.3f}  '
      f'invalid={sample["invalid_actions"]} / {sample["steps"]} steps')
print()
print('--- Trajectory ---')
for step in sample['trajectory']:
    action_str = json.dumps(step['action'])
    print(f'  step {step["step"]:2d}  reward={step["reward"]:+.3f}  '
          f'error={step["error"] or ""}  action={action_str[:80]}')

## 16 · (Optional) Save & push to Hub

In [ ]:
# Save merged model locally
trainer.save_model('./grpo_output/final')
tokenizer.save_pretrained('./grpo_output/final')
print('Model saved to ./grpo_output/final')

if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HUB_REPO_ID)
    tokenizer.push_to_hub(HUB_REPO_ID)
    print(f'Pushed to https://huggingface.co/{HUB_REPO_ID}')
else:
    print('Skipped Hub push (set PUSH_TO_HUB=True and HF_TOKEN to enable).')

## Summary

| | Baseline | Trained | Delta |
|---|---|---|---|
| T1 mean score | *(run cell 7)* | *(run cell 12)* | |
| T2 mean score | *(run cell 7)* | *(run cell 12)* | |
| Invalid action rate | *(run cell 7)* | *(run cell 12)* | |

**Key claim**: After GRPO training the agent achieves meaningfully higher episode scores on T2 (multi-order fulfillment triage), and invalid action rate drops — demonstrating it learned *structured fulfillment judgment*, not just output formatting.

The reward improvement is verified by:
1. Consistent scoring across 8 seeds (not cherry-picked)
2. Manual trajectory inspection (cell 15) confirming better warehouse/delay decisions
3. Invalid action rate going down (verifies schema compliance improved)
